# G1 Humanoid Dual-Arm Manipulation - Dataset Generation

This notebook generates synthetic manipulation motion trajectories for the **Unitree G1 humanoid robot** with dual-arm control.

**Pipeline:**
1. **Stage 1**: Isaac Lab Simulation - Generate frames with semantic segmentation and surface normals using G1 dual-arm MimicGen
2. **Stage 2**: GPU Video Encoding - Warp/CUDA kernels apply Lambertian shading
3. **Stage 3**: Cosmos Transformation - Generate diverse visual variations

**G1 Robot Configuration:**
- 29 DoF body (7 DoF per arm: 3 shoulder + 1 elbow + 3 wrist)
- Dual-arm control with Pink IK controller
- Multi-EEF MimicGen with SubTaskConstraintConfig

**Kitchen Tasks:**
- Task 1: Object Sorting (fruit classification with optional handover)
- Task 2: Rag Wiping (surface cleaning with dual-arm coordination)
- Task 3: Pot Moving (single-arm pick and place)
- Task 4: Pot Filling (placing vegetables into a pot)

**NOTE:** This notebook runs on Isaac Lab 2.3.2 (headless). No display required.

# Stage 1: Generate Motion Trajectories

## Setup Configuration

In [ ]:
# ============================================================
# [TIMING] Notebook Execution Timer - START
# ============================================================
import time
from datetime import datetime

NOTEBOOK_START_TIME = time.time()
NOTEBOOK_START_DATETIME = datetime.now()
TIMING_LOG = {}

print("=" * 60)
print(f"[G1 NOTEBOOK START] {NOTEBOOK_START_DATETIME.strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)

# ============================================================
# Setup Configuration
# ============================================================
from notebook_widgets import create_num_trials_input, create_task_input

num_envs = 1
task_widget = create_task_input()
num_trials = create_num_trials_input()

## Data Generation

Run this cell to generate G1 dual-arm manipulation demonstrations.
The standalone script handles Isaac Sim initialization and MimicGen data generation.

In [ ]:
# ============================================================
# G1 Data Generation - Standalone Mode
# ============================================================
import os
import sys

_data_gen_start = time.time()

print("=" * 60)
print("[G1] Data Generation - Standalone")
print("=" * 60)

trials = num_trials.value
task = task_widget.value

print(f"[Task] {task}")
print(f"[Trials] {trials}")
print("-" * 60)
sys.stdout.flush()

# Run standalone script with environment variables
return_code = os.system(
    f"cd /workspace/isaaclab && "
    f"NUM_TRIALS={trials} TASK='{task}' "
    f"./_isaac_sim/python.sh -u generate_data_standalone.py"
)

if return_code != 0:
    print(f"\n[ERROR] Exit code: {return_code}")
else:
    print("\n[SUCCESS] Data generation complete!")

TIMING_LOG['data_generation'] = time.time() - _data_gen_start
print(f"\n[TIMING] Data Generation: {TIMING_LOG['data_generation']/60:.2f} minutes")

# Stage 2: Video Preprocessing

Process generated frames into videos using GPU-accelerated encoding.

In [ ]:
from notebook_widgets import create_camera_input
from notebook_utils import ISAACLAB_OUTPUT_DIR

VIDEO_LENGTH = 120   # Suggested length is between 120 and 200
camera_selection = create_camera_input(ISAACLAB_OUTPUT_DIR)

In [ ]:
# ============================================================
# Video Encoding - Multi-Camera Support
# ============================================================
import os
import sys
import json
from IPython.display import Video
from notebook_utils import VIDEO_OUTPUT_DIR

_video_enc_start = time.time()

print("=" * 60)
print("[G1] Video Encoding")
print("=" * 60)

os.makedirs(VIDEO_OUTPUT_DIR, exist_ok=True)

selected_cameras = camera_selection.value

if not selected_cameras:
    print("[ERROR] No cameras selected!")
else:
    total = len(selected_cameras)
    print(f"[Cameras] {total} selected: {list(selected_cameras)}")
    print(f"[Video Length] {VIDEO_LENGTH} frames")
    print("-" * 60)
    sys.stdout.flush()

    encoded_videos = []

    for idx, camera in enumerate(selected_cameras, 1):
        print(f"\n[{idx}/{total}] Encoding {camera}...")
        sys.stdout.flush()

        video_config = {"camera": camera, "video_length": VIDEO_LENGTH}

        return_code = os.system(
            f"cd /workspace/isaaclab && "
            f"VIDEO_CONFIG='{json.dumps(video_config)}' "
            f"./_isaac_sim/python.sh -u encode_video_standalone.py"
        )

        if return_code != 0:
            print(f"[{idx}/{total}] {camera}: ERROR (code {return_code})")
        else:
            print(f"[{idx}/{total}] {camera}: SUCCESS")
            if os.path.isdir(VIDEO_OUTPUT_DIR):
                for f in sorted(os.listdir(VIDEO_OUTPUT_DIR)):
                    video_path = os.path.join(VIDEO_OUTPUT_DIR, f)
                    if f.startswith(camera) and f.endswith('.mp4') and video_path not in encoded_videos:
                        encoded_videos.append(video_path)

    print("\n" + "=" * 60)
    print(f"[COMPLETE] {len(encoded_videos)} videos encoded")
    print("=" * 60)

    for video_path in encoded_videos:
        print(f"\n{video_path}")
        display(Video(video_path, width=800))

TIMING_LOG['video_encoding'] = time.time() - _video_enc_start
print(f"\n[TIMING] Video Encoding: {TIMING_LOG['video_encoding']/60:.2f} minutes")

# Stage 3: Cosmos Transformation

Apply visual diversity transformations using NVIDIA Cosmos.

## Setup Cosmos Connection

In [ ]:
import os
COSMOS_URL = os.environ.get("COSMOS_URL", "http://host.docker.internal:18080")
print("COSMOS_URL =", COSMOS_URL)

In [ ]:
from notebook_widgets import create_variable_dropdowns, create_cosmos_params
from notebook_utils import VIDEO_OUTPUT_DIR, COSMOS_OUTPUT_DIR

# G1 kitchen prompt (mounted as stacking_prompt.toml)
prompt_manager = create_variable_dropdowns("stacking_prompt.toml")
cosmos_params = create_cosmos_params(VIDEO_OUTPUT_DIR)

## Generate with Cosmos

> **NOTE:** Generation takes around 5-10 minutes per video on a single H100 GPU.

> **Tips:**
> - Increase `Sigma Max` for stronger prompt adherence
> - Increase `Control Weight` and `Canny Strength` to reduce divergence from input

In [ ]:
# ============================================================
# Cosmos Processing - Multi-Video Support
# ============================================================
import os
import sys
from cosmos_request import process_video
from notebook_utils import VIDEO_OUTPUT_DIR, COSMOS_OUTPUT_DIR
from notebook_widgets import create_download_link
from IPython.display import Video

_cosmos_start = time.time()

print("=" * 60)
print("[G1] Cosmos Processing")
print("=" * 60)

os.makedirs(COSMOS_OUTPUT_DIR, exist_ok=True)

params = {k: w.value for k, w in cosmos_params.items()}
selected_videos = params.pop("input_video")

if not selected_videos:
    print("[ERROR] No videos selected!")
else:
    params["prompt"] = prompt_manager.prompt

    if not COSMOS_URL:
        raise ValueError("COSMOS_URL is empty. Set it via env COSMOS_URL.")

    total = len(selected_videos)
    print(f"[Videos] {total} selected")
    print(f"[Seed] {params['seed']}")
    print(f"[Control Weight] {params['control_weight']}")
    print(f"[Sigma Max] {params['sigma_max']}")
    print("-" * 60)
    sys.stdout.flush()

    processed_videos = []
    failed_videos = []

    for idx, video_name in enumerate(selected_videos, 1):
        print(f"\n[{idx}/{total}] {video_name} processing... (5-10 min)")
        sys.stdout.flush()

        video_filepath = os.path.join(VIDEO_OUTPUT_DIR, video_name)
        base_name = video_name[:-4]
        output_path = os.path.join(COSMOS_OUTPUT_DIR, f"cosmos_{base_name}_{params['seed']}.mp4")

        try:
            response = process_video(
                url=COSMOS_URL,
                video_path=video_filepath,
                output_path=output_path,
                **params,
            )

            if response is None:
                print(f"[{idx}/{total}] {video_name}: FAILED (response is None)")
                failed_videos.append(video_name)
            elif response.status_code == 200:
                print(f"[{idx}/{total}] {video_name}: SUCCESS -> {output_path}")
                processed_videos.append(output_path)
            else:
                print(f"[{idx}/{total}] {video_name}: FAILED (status {response.status_code})")
                failed_videos.append(video_name)
        except Exception as e:
            print(f"[{idx}/{total}] {video_name}: ERROR - {e}")
            failed_videos.append(video_name)

    print("\n" + "=" * 60)
    print(f"[COMPLETE] {len(processed_videos)}/{total} videos processed")
    if failed_videos:
        print(f"[FAILED] {len(failed_videos)}: {failed_videos}")
    print("=" * 60)

    for video_path in processed_videos:
        print(f"\n{video_path}")
        display(Video(video_path, width=800))
        display(create_download_link(video_path, link_text=f"Download: {os.path.basename(video_path)}"))

TIMING_LOG['cosmos_processing'] = time.time() - _cosmos_start

# ============================================================
# [TIMING] Summary
# ============================================================
NOTEBOOK_END_TIME = time.time()
TOTAL_ELAPSED = NOTEBOOK_END_TIME - NOTEBOOK_START_TIME

print("\n" + "=" * 60)
print("[G1 NOTEBOOK COMPLETE] Execution Time Summary")
print("=" * 60)
print(f"Start: {NOTEBOOK_START_DATETIME.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"End:   {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("-" * 60)
for step, elapsed in TIMING_LOG.items():
    print(f"  {step:20s}: {elapsed/60:6.2f} minutes")
print("-" * 60)
print(f"  {'TOTAL':20s}: {TOTAL_ELAPSED/60:6.2f} minutes")
print("=" * 60)